<a href="https://colab.research.google.com/github/HLZHarry/LLM-Practice/blob/main/ch04/Ch04_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

# Load our data
data = load_dataset("rotten_tomatoes")
data

In [ ]:
data["train"][0, -1]

In [ ]:
from transformers import pipeline

# Path to our HF model
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Load model into pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device="cuda:0"
)

In [ ]:
# Check what one output looks like
output_sample = pipe(data["test"]["text"][2])
print(output_sample)

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
  # print(output)
  scores = {d["label"]: d["score"] for d in output}

  negative_score = scores["negative"]
  positive_score = scores["positive"]
  assignment = np.argmax([negative_score, positive_score])
  # print(negative_score, positive_score, assignment)
  y_pred.append(assignment)

## What Does This Code Do?

This code runs **sentiment classification** on the entire test dataset and stores the predictions.

---

## Breaking Down `tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"]))`

### `KeyDataset(data["test"], "text")`
- A HuggingFace utility that **extracts just the `"text"` column** from the test dataset
- Instead of passing the whole dataset object, it feeds only the text strings to the pipeline
- Efficient — streams data one by one instead of loading everything into memory

### `pipe(...)`
- Runs the **text-classification pipeline** on every text sample in the dataset
- Returns an output for each sample (a list of label/score dicts)
- Processes samples lazily (one at a time) for memory efficiency

### `tqdm(..., total=len(data["test"]))`
- Wraps the pipeline output in a **progress bar**
- `total=len(data["test"])` tells tqdm the total number of samples so it can show percentage completion and estimated time remaining

---

## The Full Loop
```python
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    negative_score = output[0]["score"]   # score for "negative" label
    positive_score = output[2]["score"]   # score for "positive" label
    assignment = np.argmax([negative_score, positive_score])  # picks the higher score → 0=negative, 1=positive
    y_pred.append(assignment)             # stores the prediction
```

- `output[0]["score"]` — extracts the **negative** label score
- `output[2]["score"]` — extracts the **positive** label score (index 2 because there are 3 labels: negative, neutral, positive)
- `np.argmax([negative_score, positive_score])` — returns `0` if negative is higher, `1` if positive is higher
- `y_pred.append(assignment)` — builds a list of predictions for the entire test set

---

## Final Result
`y_pred` is a list like `[1, 0, 1, 1, 0, ...]` where:
- `0` = predicted **negative**
- `1` = predicted **positive**

In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names = ["Negative Review", "Positive Review"]
  )
  print(performance)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

In [ ]:
train_embeddings.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

In [ ]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# Average the embeddings of all documents in each target label
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values

# Find the best matching embeddings between evaluation documents and target embeddings
sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

# Evaluate the model
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# Create embeddings for our labels
label_embeddings = model.encode(["A negative review",  "A positive review"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
import openai

client = openai.OpenAI(api_key="")

In [ ]:
def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    """Generate an output based on a prompt and an input document."""
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
            },
        {
            "role": "user",
            "content":   prompt.replace("[DOCUMENT]", document)
            }
    ]
    chat_completion = client.chat.completions.create(
      messages=messages,
      model=model,
      temperature=0
    )
    return chat_completion.choices[0].message.content

In [ ]:
# Define a prompt template as a base
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# Predict the target using GPT
document = "unpretentious , charming , quirky , original"
chatgpt_generation(prompt, document)

In [ ]:
predictions = [chatgpt_generation(prompt, doc) for doc in tqdm(data["test"]["text"])]

In [ ]:
y_pred = [int(pred) for pred in predictions]

# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)